# Module 4: Improving Agentic Workflow with Follow-up Questions (Optional)

In this optional module, you'll learn how to improve your AI agent's workflow by implementing **follow-up questions** for ambiguous user queries using the `AskUserQuestion` tool.

## Learning Objectives

By the end of this module, you will:
- Understand the `AskUserQuestion` tool for structured clarification
- Implement interactive clarification workflows using `can_use_tool` callback
- Deploy agents with clarification support to AgentCore
- Handle multi-turn conversations with session resumption

## Why Follow-up Questions?

Natural language queries are often ambiguous. For example:
- "Show me the top students" - Top by what metric? GPA? Enrollment? Scholarships?
- "What's the enrollment trend?" - For which department? What time period?
- "How are students performing?" - Academic performance? Financial? Attendance?

Instead of guessing, a well-designed agent should **ask for clarification** before executing queries.

## Prerequisites

Before starting this module, ensure:
- Module 0 setup is complete (environment configured)
- Module 1 completed (understanding of local agent execution)
- Module 2 completed (understanding of AgentCore deployment)

## Step 1: Setup Environment

In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path
from dotenv import load_dotenv

# Set up paths
workshop_root = Path("..").resolve()
module_dir = Path(".").resolve()
sys.path.insert(0, str(workshop_root))

# Load environment
load_dotenv(workshop_root / ".env")

# CRITICAL: Ensure Claude Agent SDK uses Bedrock
os.environ["CLAUDE_CODE_USE_BEDROCK"] = "1"

# Install workshop dependencies (including claude_agent_sdk)
print("Installing workshop dependencies...")
!uv pip install -e {workshop_root} --quiet
print("[OK] Dependencies installed")

print(f"\nWorkshop root: {workshop_root}")
print(f"Module directory: {module_dir}")
print(f"AWS Region: {os.getenv('AWS_REGION')}")
print(f"Using Bedrock: {os.getenv('CLAUDE_CODE_USE_BEDROCK')}")

## Step 2: Understanding the AskUserQuestion Tool

The Claude Agent SDK provides a built-in `AskUserQuestion` tool that enables structured clarification questions. This is more robust than text-based approaches because:

1. **Structured Format**: Questions have headers, descriptions, and options with labels
2. **Type Safety**: Options are typed with `label` and `description` fields
3. **Multi-select Support**: Questions can allow single or multiple selections
4. **Callback Pattern**: Uses `can_use_tool` callback for handling responses

### AskUserQuestion Tool Schema

```json
{
  "questions": [
    {
      "header": "Student Ranking Criteria",
      "question": "How would you like to define 'top students'?",
      "options": [
        {"label": "By GPA", "description": "Students with highest cumulative GPA"},
        {"label": "Dean's List", "description": "Students with GPA >= 3.75"},
        {"label": "By Pass Rate", "description": "Students who passed most courses"}
      ],
      "multiSelect": false
    }
  ]
}
```

### Implementation Pattern

The key components for implementing clarification:

1. **`can_use_tool` callback** - Intercepts AskUserQuestion tool calls
2. **`HookMatcher` with `PreToolUse`** - Keeps the stream open for callback
3. **`PermissionResultAllow`** - Returns with collected answers

In [ ]:
# Let's examine the key implementation pattern
print("Key Implementation Pattern for AskUserQuestion Tool")
print("=" * 60)
print("""
from claude_agent_sdk.types import (
    HookMatcher,
    PermissionResultAllow,
    PermissionResultDeny,
    ToolPermissionContext,
)

# 1. Define the can_use_tool callback
async def can_use_tool(
    tool_name: str, input_data: dict, context: ToolPermissionContext
) -> PermissionResultAllow | PermissionResultDeny:
    
    # Handle AskUserQuestion tool
    if tool_name == "AskUserQuestion":
        # Collect user answers interactively
        answers = collect_user_answers(input_data["questions"])
        
        # Return with both questions and answers
        return PermissionResultAllow(
            updated_input={
                "questions": input_data["questions"],
                "answers": answers
            }
        )
    
    # Auto-approve other tools
    return PermissionResultAllow(updated_input=input_data)

# 2. Hook to keep stream open
async def can_use_tool_hook(input_data, tool_use_id, context):
    return {"continue_": True}

# 3. Configure agent options
options = ClaudeAgentOptions(
    allowed_tools=["AskUserQuestion", ...],
    can_use_tool=can_use_tool,
    hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[can_use_tool_hook])]},
    ...
)
""")
print("\n[OK] This pattern enables interactive clarification in the agent")

## Step 3: Review the Follow-up Agent Code

Let's examine the follow-up agent implementation in `followup_agent.py`.

In [ ]:
# Read and display key sections of the followup_agent.py
followup_agent_path = module_dir / "followup_agent.py"

with open(followup_agent_path, 'r') as f:
    content = f.read()

# Show the handle_ask_user_question function
print("[FILE] Key Function: handle_ask_user_question")
print("=" * 60)

# Extract and display the function
import re
match = re.search(r'(async def handle_ask_user_question.*?)\n\nasync def run', content, re.DOTALL)
if match:
    print(match.group(1))
else:
    print("Function not found - please check followup_agent.py")

In [ ]:
# Show the agent configuration with AskUserQuestion
print("[FILE] Agent Configuration with AskUserQuestion Tool")
print("=" * 60)

# Extract allowed_tools and can_use_tool config
match = re.search(r'(# Configure options.*?max_turns=30\s*\))', content, re.DOTALL)
if match:
    print(match.group(1))
else:
    print("Configuration not found - please check followup_agent.py")

## Step 4: Test the Local Follow-up Agent

Since interactive `input()` doesn't work well in Jupyter notebooks, we'll run the agent from the terminal.

**Open a terminal in the workshop root directory and run:**

```bash
# Test with an ambiguous query that should trigger clarification
uv run python module-4-improve-agentic-workflow/followup_agent.py "Show me the top lecturers"

# The agent will:
# 1. Load the appropriate skill
# 2. Detect ambiguity in "top lecturers"
# 3. Use AskUserQuestion tool to ask for clarification
# 4. Wait for your input
# 5. Continue with the analysis
```

### Expected Output Flow:

```
================================================================================
STUDENT ANALYTICS AI AGENT
================================================================================
Request ID: <uuid>

User Query: Show me the top lecturers

--------------------------------------------------------------------------------
Claude Agent SDK Session ID: <session-id>

Loading skill: enrollment

Before I query the data, I'd like to clarify what you mean by "top lecturers":

❓ Claude is asking 2 clarification question(s)...

────────────────────────────────────────────────────────────────────────────────
📋 Claude needs clarification:
────────────────────────────────────────────────────────────────────────────────

Ranking by: What metric would you like to use to rank the lecturers?
  1. Most students enrolled - Lecturers with the highest number of students currently enrolled in their courses
  2. Most courses taught - Lecturers teaching the highest number of courses
  (Enter a number, or type your own answer)
Your choice: 1

Instructor type: Should I include all instructors or only those with 'Lecturer' rank?
  1. All instructors - Include all ranks: Professor, Associate Professor, Assistant Professor, Lecturer, Adjunct
  2. Only Lecturers - Filter to only instructors with rank = 'Lecturer'
  (Enter a number, or type your own answer)
Your choice: 1
────────────────────────────────────────────────────────────────────────────────

--------------------------------------------------------------------------------
... agent continues with SQL query ...
```

In [ ]:
# Generate the command to run in terminal
print("[TERMINAL] Run this command in your terminal:")
print("=" * 60)
print(f"\ncd {workshop_root}")
print(f"\nuv run python module-4-improve-agentic-workflow/followup_agent.py \"Show me the top lecturers\"")
print("\n" + "=" * 60)

## Step 5: AgentCore Deployment with Session Resumption

For AgentCore deployment, the clarification flow works differently:

1. **First Invocation**: Agent detects ambiguity and outputs structured JSON with questions
2. **Client Script**: `invoke_agentcore.py` parses questions and prompts user
3. **Follow-up Invocation**: Script sends answers with `claude_agent_sdk_session_id` to resume
4. **Session Resumption**: Agent continues from where it left off

### Key Features:

- **Session Preservation**: `claude_agent_sdk_session_id` maintains conversation context
- **Multi-Round Support**: Handles multiple rounds of clarification automatically
- **Structured Output**: JSON format for reliable parsing

In [ ]:
# Review the AgentCore agent configuration
agentcore_path = module_dir / "followup_agent_agentcore.py"

with open(agentcore_path, 'r') as f:
    agentcore_content = f.read()

print("[FILE] AgentCore Session Resumption Configuration")
print("=" * 60)

# Show the session resumption logic
match = re.search(r'(# Extract optional Claude Agent SDK Session ID.*?logger\.info.*?session.*?\))', agentcore_content, re.DOTALL)
if match:
    print(match.group(1))
else:
    print("Session resumption logic - check followup_agent_agentcore.py")

print("\n" + "=" * 60)
print("[FILE] Adding Resume Parameter")
print("=" * 60)

match = re.search(r'(# Add resume parameter.*?logger\.info.*?session ID.*?\))', agentcore_content, re.DOTALL)
if match:
    print(match.group(1))
else:
    print("Resume parameter logic - check followup_agent_agentcore.py")

## Step 6: Configure and Deploy to AgentCore

The follow-up agent is configured as a separate agent in `.bedrock_agentcore.yaml` with:
- Agent name: `student_analytics_agent_followup`
- Entrypoint: `main_followup.py`

This allows deploying the follow-up agent alongside the basic agent from Module 2.

In [ ]:
import yaml

# Check and display the AgentCore configuration
config_file = workshop_root / ".bedrock_agentcore.yaml"

if config_file.exists():
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    
    print("[CONFIG] Current AgentCore Configuration")
    print("=" * 60)
    
    # Check if follow-up agent is configured
    if 'student_analytics_agent' in config.get('agents', {}):
        followup_config = config['agents']['student_analytics_agent']
        print("[OK] student_analytics_agent agent found in configuration!")
        print(f"\n  Agent Name: {followup_config.get('name')}")
        print(f"  Entrypoint: {followup_config.get('entrypoint')}")
        print(f"  Platform: {followup_config.get('platform')}")
        print(f"  Region: {followup_config['aws'].get('region')}")
        
        deployed = followup_config.get('bedrock_agentcore', {})
        if deployed.get('agent_id'):
            print(f"  Agent ID: {deployed.get('agent_id')}")
            print(f"  Status: Previously deployed")
        else:
            print(f"  Status: Not yet deployed")
    else:
        print("[WARNING] agent not found in configuration")
        print("Run Module 2 first, then re-run this module.")
else:
    print("[ERROR] .bedrock_agentcore.yaml not found")
    print("Please run Module 2 first to configure AgentCore.")

In [ ]:
with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

# Check and fix entrypoint if needed (might be main_observable.py from Module 3b)
current_entrypoint = config['agents']['student_analytics_agent'].get('entrypoint', '')
if current_entrypoint != 'main_followup.py':
    config['agents']['student_analytics_agent']['entrypoint'] = 'main_followup.py'
    with open(config_file, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
    print(f"📝 Updated entrypoint from '{current_entrypoint}' to 'main_followup.py' for Module 4")
else:
    print("✅ Configuration already set for Module 4 (entrypoint: main_followup.py)")

In [ ]:
# Update Dockerfile entry point
import re

dockerfile_path = workshop_root / "Dockerfile"

# Check if Dockerfile exists
if not dockerfile_path.exists():
    print("❌ Dockerfile not found!")
    print("   Please run Module 2 first to generate the Dockerfile.")
else:
    print("[OK] Dockerfile found")
    
    # Read current content
    with open(dockerfile_path, 'r') as f:
        dockerfile_content = f.read()
    
    # Find and replace CMD line with any Python file
    # Pattern matches: CMD ["python", "any_file.py"]
    pattern = r'CMD \[.*"python".*\.py"\]'
    replacement = 'CMD ["python", "main_followup.py"]'
    
    if re.search(pattern, dockerfile_content):
        # Extract current file for logging
        current_match = re.search(r'CMD \["python", "([^"]+\.py)"\]', dockerfile_content)
        current_file = current_match.group(1) if current_match else "unknown"
        
        # Replace with main_followup.py
        updated_content = re.sub(pattern, replacement, dockerfile_content)
        
        # Write back
        with open(dockerfile_path, 'w') as f:
            f.write(updated_content)
        
        if current_file == "main_followup.py":
            print("✅ Dockerfile already configured to run main_followup.py")
        else:
            print(f"📝 Updated Dockerfile CMD from '{current_file}' to 'main_followup.py'")
    else:
        print("[WARNING] Could not find CMD pattern in Dockerfile")
        print("Please check the Dockerfile manually")

In [ ]:
# Verify main_followup.py exists
main_followup_path = workshop_root / "main_followup.py"

if main_followup_path.exists():
    print("[OK] main_followup.py entrypoint exists")
    print("\nContents:")
    print("=" * 60)
    with open(main_followup_path, 'r') as f:
        print(f.read())
else:
    print("[ERROR] main_followup.py not found")
    print("Please ensure the file exists at the workshop root.")

In [ ]:
# Deploy the follow-up agent to AgentCore
print("[DEPLOY] Deploying follow-up agent to AgentCore...")
print("=" * 60)
print("This will deploy the agent.")
print("Estimated time: 5-10 minutes for first deployment")
print("=" * 60)

result = subprocess.run(
    ["agentcore", "deploy"],
    cwd=str(workshop_root),
    capture_output=True,
    text=True,
    timeout=600
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print("\n[OK] Deployment successful!")
else:
    print(f"\n[ERROR] Deployment failed with return code: {result.returncode}")

In [ ]:
# Check deployment status
print("[STATUS] Checking follow-up agent status...")
print("=" * 60)

result = subprocess.run(
    ["agentcore", "status"],
    cwd=str(workshop_root),
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    if "not yet deployed" in result.stderr.lower() or "not deployed" in result.stdout.lower():
        print("Agent not yet deployed. Run the deployment cell above.")
    else:
        print("Error checking status:")
        print(result.stderr or result.stdout)

## Step 7: Test AgentCore with Multi-Round Clarification

Use the `invoke_agentcore.py` script which handles multi-round clarification automatically.

**Open a terminal and run:**

```bash
# Local dev mode (First run the agent locally with `agentcore dev`)
uv run python scripts/invoke_agentcore.py --dev "Show me the top lecturers"

# AWS hosted mode (after deployment)
uv run python scripts/invoke_agentcore.py "Show me the top lecturers"
```

In [ ]:
# Generate the commands to run in terminal
print("[TERMINAL] Run these commands in your terminal:")
print("=" * 60)
print(f"\ncd {workshop_root}")
print("\n# AWS hosted mode (after deployment):")
print('uv run python scripts/invoke_agentcore.py "Show me the top lecturers"')
print("\n" + "=" * 60)

## Key Takeaways

### Benefits of AskUserQuestion Tool

| Benefit | Description |
|---------|-------------|
| **Structured** | Questions have typed options with labels and descriptions |
| **Robust** | Uses callback pattern instead of text parsing |
| **Flexible** | Supports single and multi-select questions |
| **Resumable** | Session ID enables multi-turn conversations |

### Implementation Patterns

| Environment | Pattern | Key Components |
|-------------|---------|----------------|
| **Local/CLI** | Interactive input() | `can_use_tool` callback, `HookMatcher` |
| **AgentCore** | JSON output + re-invoke | Session resumption via `resume` parameter |
| **Web App** | API polling | Parse JSON, send follow-up with session ID |

### Best Practices

1. **Provide Clear Options** - Use descriptive labels and descriptions
2. **Limit Questions** - Don't overwhelm users (2-4 questions max)
3. **Allow Custom Answers** - Accept free-text in addition to options
4. **Preserve Context** - Use session IDs for multi-turn conversations
5. **Handle Errors** - Gracefully handle missing or invalid answers

## Summary

In this module, you learned:

- How to use the `AskUserQuestion` tool for structured clarification
- How to implement `can_use_tool` callback for handling tool permissions
- How session resumption works in AgentCore with `claude_agent_sdk_session_id`
- How to deploy a separate follow-up agent with `agentcore deploy --agent student_analytics_agent_followup`

This completes the optional Module 4. You now have a more robust agent that can handle ambiguous queries gracefully!

---

*Workshop: Build Agentic AI Applications with Claude Agent SDK and Amazon Bedrock AgentCore*